# Detergent soluble / insoluble unenriched proteomics (HK3-124)

TMT-16plex unenriched proteomics of detergent-soluble and detergent-insoluble fractions
across three T cell states (D2, D8A, D8C), two replicates each.

This notebook does the computation and writes CSVs to `data/solubility/`.
All publication figures are drawn in `solubility_visualization.Rmd`.

See `README.md` in this folder for the normalization equation and its caveats.

In [ ]:
# %load_ext autoreload
# %autoreload 2

import shutil
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

from solubility import *

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

In [ ]:
data_dir = Path("../../data/solubility")
input_dir = data_dir / "01_input"
normalized_dir = data_dir / "02_normalized"
results_dir = data_dir / "03_results"
volcano_dir = results_dir / "volcano_plots"

for d in [input_dir, normalized_dir, results_dir, volcano_dir]:
    d.mkdir(parents=True, exist_ok=True)

## Snapshot the source

The source lives in Dropbox, where it can be edited or dematerialised. Copy it locally on
first run and record its checksum so the analysis is reproducible from the repo.

In [ ]:
local_copy = input_dir / SOURCE_CSV.name
if not local_copy.exists():
    shutil.copy2(SOURCE_CSV, local_copy)

digest = hashlib.sha256(local_copy.read_bytes()).hexdigest()
print(f"source : {SOURCE_CSV}")
print(f"local  : {local_copy}")
print(f"sha256 : {digest}")

## Load

Drops the four empty `No_*` TMT channels, renames samples to
`<state>-<fraction>_<replicate>`, and converts zeros (TMT non-detections) to NaN.

In [ ]:
channel_ratios = load_channel_ratios(local_copy)

print(f"proteins : {len(channel_ratios)}")
print(f"samples  : {len(sample_columns())}")
print(
    f"complete across all channels : {channel_ratios[sample_columns()].notna().all(axis=1).sum()}"
)
print("\nnon-detections (zeros -> NaN) per channel:")
print(channel_ratios[sample_columns()].isna().sum().to_string())
channel_ratios.head()

## Normalize

Equal protein mass from each fraction was TMT-labelled, so the raw channel ratio reports a
protein's share *within its own fraction*. Dividing the insoluble channels by the per-state
Final dilution factor puts both fractions on a common per-cell basis:

$$F_g = \frac{C_{sol,g} \cdot V_{sol}}{C_{insol,g} \cdot V_{insol}}, \qquad r'_{insol} = r_{insol} / F_g$$

In [ ]:
# Confirm the factors reproduce from the recorded concentrations and volumes.
for state in STATES:
    mass = {f: PROTEIN_CONC[state][f] * FRACTION_VOLUME_UL[f] / 1000 for f in FRACTIONS}
    derived = mass["Soluble"] / mass["Insoluble"]
    print(
        f"{state:4s} soluble {mass['Soluble']:.4f} mg  insoluble {mass['Insoluble']:.4f} mg"
        f"   F = {derived:.6f}  (recorded {FINAL_DILUTION[state]:.6f})"
    )
    assert np.isclose(derived, FINAL_DILUTION[state]), state

In [ ]:
normalized = normalize_fractions(channel_ratios)
normalized.to_csv(normalized_dir / "normalized_channel_ratios.csv", index=False)

print("median log2(Insoluble / Soluble) per state:")
for state in STATES:
    for label, table in [("raw", channel_ratios), ("normalized", normalized)]:
        insol = table[group_columns(f"{state}-Insoluble")].mean(axis=1)
        sol = table[group_columns(f"{state}-Soluble")].mean(axis=1)
        value = np.log2(insol / sol).replace([np.inf, -np.inf], np.nan).median()
        print(f"  {state:4s} {label:11s} {value:6.2f}")

## PCA

Run on both inputs. The per-fraction constant is a pure translation on the log2 scale, so it
loads onto PC1 and inflates it without adding biological signal - PC2, which separates the
cell states, is essentially unchanged. The un-normalized PCA is the one to show.

In [ ]:
pca_inputs = {
    "unnormalized": channel_ratios,
    "normalized": normalized,
}

for name, table in pca_inputs.items():
    loadings, scores, percent, n_proteins = get_pca_plot(
        table[["uniprot"] + sample_columns()],
        "uniprot",
        results_dir / f"pca_{name}",
    )
    explained = "  ".join(
        f"{r.principal_component}={r.percent_explained}%" for r in percent.itertuples()
    )
    print(f"{name:13s} {n_proteins} proteins   {explained}")

In [ ]:
# QC: PC1 separates fraction, PC2 separates state. Note that the two D2 insoluble
# replicates are markedly less concordant than any other pair.
pd.read_csv(results_dir / "pca_unnormalized" / "pca_results.csv")[
    ["channel_name", "condition", "state", "fraction", "PC1", "PC2"]
].round(1)

## Volcano statistics

Nine comparisons: three fraction contrasts, log2(Insoluble / Soluble) within each state, and
six state contrasts (D8A/D2, D8C/D2, D8C/D8A) within each fraction.

Statistics follow the repo: independent two-sample t-test on linear channel ratios
(`equal_var=True`, `nan_policy="omit"`), significance called on raw p < 0.05 **and**
|log2FC| > log2(1.5). BH-adjusted p is stored but not used for calling.

Replicate columns are selected by explicit name rather than the `filter(like=...)` substring
matching used elsewhere - a bare `"D2"` matches all four D2 columns across *both* fractions.

In [ ]:
long_volcano_df, protein_counts = run_comparisons(normalized, volcano_dir)

protein_counts[
    [
        "condition",
        "control_condition",
        "n_input",
        "n_tested",
        "n_dropped_group_too_small",
        "n_dropped_other",
        "median_log2_FC",
        "n_significant_up",
        "n_significant_down",
    ]
]

### Protein count audit

Every protein in the input must be either tested or accounted for by a drop reason.

In [ ]:
reconciled = (
    protein_counts["n_tested"]
    + protein_counts["n_dropped_group_too_small"]
    + protein_counts["n_dropped_other"]
)
assert (reconciled == protein_counts["n_input"]).all()
assert len(long_volcano_df) == protein_counts["n_tested"].sum()
assert protein_counts["n_p_zero"].sum() == 0
assert np.isfinite(long_volcano_df["neg_log10_pval"]).all()
assert np.isfinite(long_volcano_df["log2_FC"]).all()

print(
    f"all {len(protein_counts)} comparisons reconcile against {protein_counts['n_input'].iloc[0]} input proteins"
)
print(f"long volcano table: {len(long_volcano_df)} rows")
print(
    f"log2_FC range: {long_volcano_df['log2_FC'].min():.2f} to {long_volcano_df['log2_FC'].max():.2f}"
)
print(f"neg_log10_pval max: {long_volcano_df['neg_log10_pval'].max():.2f}")

### Caveat: n = 2

The replicates are TMT channel duplicates within a single MS run, not independent biological
replicates, so the t-test has 2 degrees of freedom. The soluble replicates in particular are
very tight, which collapses the pooled SD and makes the p-value filter permissive. Read the
volcanoes on fold change; do not over-read significance.

In [ ]:
print("within-group median |log2(rep1 / rep2)|:")
for state in STATES:
    for fraction in FRACTIONS:
        a, b = group_columns(f"{state}-{fraction}")
        spread = np.log2(normalized[a] / normalized[b]).abs().median()
        print(f"  {state}-{fraction:10s} {spread:.3f}")

print("\nfraction of tested proteins with raw p < 0.05:")
for (cond, ctrl), grp in long_volcano_df.groupby(
    ["condition", "control_condition"], sort=False
):
    frac = (grp["p_value"] < 0.05).mean()
    print(f"  {cond} vs {ctrl:16s} {frac * 100:5.1f}%")

### Cross-check against the existing Excel workbook

For the eight proteins analysed by hand, the pipeline's normalized log2(I/S) must equal
log2 of the workbook's raw `I/S` ratio minus log2(F).

In [ ]:
spot_check = ["MAP2K3", "MAP2K4", "LONP1", "HSPA9", "DNAJA3", "NDUFA9", "VDAC1", "GZMB"]

rows = []
for state in STATES:
    insol = channel_ratios[group_columns(f"{state}-Insoluble")].mean(axis=1)
    sol = channel_ratios[group_columns(f"{state}-Soluble")].mean(axis=1)
    raw_ratio = insol / sol
    for protein in spot_check:
        mask = channel_ratios["protein"] == protein
        if not mask.any():
            continue
        rows.append(
            {
                "protein": protein,
                "state": state,
                "raw_I/S": float(raw_ratio[mask].iloc[0]),
                "normalized_log2(I/S)": float(
                    np.log2(raw_ratio[mask].iloc[0]) - np.log2(FINAL_DILUTION[state])
                ),
            }
        )

spot_df = pd.DataFrame(rows).pivot(index="protein", columns="state")
spot_df.round(3)

## GSEA

Pre-ranked GSEA (gseapy) on MSigDB C5 GO biological process, for the three insoluble/soluble
fraction contrasts, following `notebooks/01_whole_proteome`: 10000 permutations, seed 42.

Proteins are ranked by log2(Insoluble / Soluble), so **a positive NES means the set is
enriched among proteins gaining intensity in the insoluble fraction**. GSEA is rank-based, so
the large global offset of these contrasts (median log2(I/S) of -3 to -4) does not affect the
result - only the relative order matters.

Takes about 10 minutes.

In [ ]:
gsea_results = run_fraction_gsea(long_volcano_df, results_dir / "gsea")

# Direction check: the leading-edge genes of the top positive-NES sets must sit above the
# overall median log2(I/S), and those of the top negative-NES sets below it.
for state, report in gsea_results.items():
    fold_changes = (
        long_volcano_df[
            (long_volcano_df["condition"] == f"{state}-Insoluble")
            & (long_volcano_df["control_condition"] == f"{state}-Soluble")
        ]
        .groupby("protein")["log2_FC"]
        .mean()
    )
    overall = fold_changes.median()
    significant = report[report["FDR q-val"] < 0.01]

    def leading_edge_median(row):
        genes = [
            g for g in str(row["Lead_genes"]).split(";") if g in fold_changes.index
        ]
        return fold_changes[genes].median()

    up = significant.nlargest(5, "NES").apply(leading_edge_median, axis=1)
    down = significant.nsmallest(5, "NES").apply(leading_edge_median, axis=1)
    assert (up > overall).all() and (down < overall).all(), state
    print(
        f"{state}: overall median log2(I/S) {overall:+.2f} | "
        f"top +NES leading edge {up.min():+.2f} to {up.max():+.2f} | "
        f"top -NES {down.min():+.2f} to {down.max():+.2f}"
    )

print("\npositive NES = enriched among proteins up in the insoluble fraction")

# Revision run: HK3-129 + HK3-130

Two further MS runs acquired for the revisions, combined into one channel-ratio table. Each
file carries the full D2/D8A/D8C x soluble/insoluble design as TMT channel duplicates, so a
state x fraction group has four channels: **two donors** (`d1` = HK3-129, `d2` = HK3-130) x
two duplicates.

Three things differ from HK3-124 above:

- **The replicate unit is the donor.** HK3-124's two replicates were channel duplicates
  within one run; here they are independent preparations.
- **No normalization.** No soluble/insoluble protein concentrations were recorded for these
  runs, so there is no Final dilution factor to divide out. The fraction contrasts keep a
  large negative global offset - read it as a global shift, not as regulation.
- **A two-donor filter** replaces the pipeline's own replicate filter, which is unusable
  here (see below).

In [ ]:
rev_cfg = REVISION
rev_data_dir = Path("../../data/solubility/revision")
rev_input_dir = rev_data_dir / "01_input"
rev_results_dir = rev_data_dir / "03_results"
rev_volcano_dir = rev_results_dir / "volcano_plots"
rev_scatter_dir = rev_results_dir / "scatterplots"
rev_barplot_dir = rev_results_dir / "barplots"

for d in [rev_input_dir, rev_results_dir, rev_volcano_dir, rev_scatter_dir, rev_barplot_dir]:
    d.mkdir(parents=True, exist_ok=True)

## Snapshot the source

The channel-level file, not the combined-condition files. A typo in the pipeline's
`parameter_dict.json` - `"D\n2-Insoluble"`, and `Insoluble` capitalised against the lowercase
channel names - left all three insoluble columns of `01_*` and `04_*` empty, so those tables
carry only the soluble conditions.

In [ ]:
rev_local = rev_input_dir / rev_cfg.source.name
if not rev_local.exists():
    shutil.copy2(rev_cfg.source, rev_local)

print(f"source : {rev_cfg.source}")
print(f"local  : {rev_local}")
print(f"sha256 : {hashlib.sha256(rev_local.read_bytes()).hexdigest()}")

## Load

Renames samples to `<state>-<fraction>_<donor>_<duplicate>`, drops the unused `No_*` TMT
slots, and converts zeros (TMT non-detections) to NaN.

In [ ]:
rev_channel_ratios = load_channel_ratios(rev_local, cfg=rev_cfg)
rev_samples = sample_columns(rev_cfg)

print(f"proteins : {len(rev_channel_ratios)}")
print(f"samples  : {len(rev_samples)}")
print("\nnon-detections (zeros -> NaN) per channel:")
print(rev_channel_ratios[rev_samples].isna().sum().to_string())
rev_channel_ratios.head()

## Two-donor filter

A protein is kept only if it was quantified in **both donors in every one of the six**
state x fraction groups; a donor counts as quantified when at least one of its two TMT
duplicates is not NaN. A protein seen in only one MS file has no replication at all,
whatever its channel count suggests.

This replaces the pipeline's `min4rep` files, which are asymmetric: they dropped exactly the
411 proteins absent from HK3-130 but kept the 317 absent from HK3-129.

The cell below also answers the question this analysis was asked to check first - whether the
filter costs any of the eight proteins of interest.

In [ ]:
rev_filtered, rev_audit = two_donor_filter(
    rev_channel_ratios, cfg=rev_cfg, out_dir=rev_results_dir
)
rev_filtered.to_csv(rev_results_dir / "channel_ratios_two_donor_filtered.csv", index=False)

poi_audit = rev_audit[rev_audit["protein"].isin(PROTEINS_OF_INTEREST)]
assert poi_audit["kept"].all(), "a protein of interest was dropped by the two-donor filter"
assert len(poi_audit) == len(PROTEINS_OF_INTEREST)

print("\ndonors seen per group, proteins of interest:")
poi_audit.drop(columns=["uniprot", "description"])

## PCA

Un-normalized only: there is nothing to normalize here. Complete cases across all 24 channels.

In [ ]:
rev_loadings, rev_scores, rev_percent, rev_n_proteins = get_pca_plot(
    rev_filtered[["uniprot"] + rev_samples],
    "uniprot",
    rev_results_dir / "pca_unnormalized",
)
explained = "  ".join(
    f"{r.principal_component}={r.percent_explained}%" for r in rev_percent.itertuples()
)
print(f"{rev_n_proteins} complete proteins   {explained}")

pd.read_csv(rev_results_dir / "pca_unnormalized" / "pca_results.csv")[
    ["channel_name", "condition", "state", "fraction", "PC1", "PC2"]
].round(1)

## Volcano statistics

The same nine comparisons as HK3-124, on four channels per group.

**Caveat on the degrees of freedom.** Those four channels are two donors x two TMT
duplicates, and the duplicates are technical, so `ttest_ind` with df = 6 treats four values
as four independent observations when there are really two. p-values are anti-conservative.
This is a deliberate choice for this run; as with HK3-124, read the volcanoes primarily on
fold change.

In [ ]:
rev_long_df, rev_counts = run_comparisons(rev_filtered, rev_volcano_dir, cfg=rev_cfg)

rev_counts[
    [
        "condition",
        "control_condition",
        "n_input",
        "n_tested",
        "n_dropped_group_too_small",
        "n_dropped_other",
        "median_log2_FC",
        "n_significant_up",
        "n_significant_down",
    ]
]

### Protein count audit

In [ ]:
rev_reconciled = (
    rev_counts["n_tested"]
    + rev_counts["n_dropped_group_too_small"]
    + rev_counts["n_dropped_other"]
)
assert (rev_reconciled == rev_counts["n_input"]).all()
assert len(rev_long_df) == rev_counts["n_tested"].sum()
assert rev_counts["n_p_zero"].sum() == 0
assert np.isfinite(rev_long_df["neg_log10_pval"]).all()
assert np.isfinite(rev_long_df["log2_FC"]).all()

# Every protein of interest must survive into every comparison.
poi_per_comparison = (
    rev_long_df[rev_long_df["protein"].isin(PROTEINS_OF_INTEREST)]
    .groupby(["condition", "control_condition"])
    .size()
)
assert (poi_per_comparison == len(PROTEINS_OF_INTEREST)).all()

print(
    f"all {len(rev_counts)} comparisons reconcile against "
    f"{rev_counts['n_input'].iloc[0]} input proteins"
)

## Scatterplots

Two families of log2FC-against-log2FC plots, both classified against a 1.5-fold band around
identity:

- **state vs state** - the fraction contrast log2(Insoluble/Soluble) of one state against
  another.
- **soluble vs insoluble** - the same state contrast measured in each fraction.

Note that for a matched pair the two are algebraically the same difference:

```
(I_D8C - S_D8C) - (I_D2 - S_D2)  ==  (I_D8C - I_D2) - (S_D8C - S_D2)
```

so a matched pair shares its `delta`, and therefore its off-band protein set, exactly. They
are two rotations of one quantity: the axes, the Pearson r and what the picture emphasises
differ, the classification does not. The cell below asserts that identity rather than leaving
the duplicated counts looking like a bug.

### Percent insoluble

The state-vs-state family carries two extra columns, `percent_insoluble_x` and
`percent_insoluble_y`. A state's soluble and insoluble shares sum to 100% by construction, so
log2(Insoluble/Soluble) is only a reparameterisation of "% insoluble" - the same contrast on a
less readable scale than the barplot below, which is already in percent. The Rmd draws the
state-vs-state family both ways from this one table, into separate figure folders.

The 1.5-fold band is unchanged: on percent axes `|delta| > log2(1.5)` is the odds curve
`y/(100-y) = 1.5 * x/(100-x)`, which `run_scatter` asserts the percent columns reproduce
exactly. The off-band protein set and every count in `scatter_summary.csv` are therefore
identical between the two versions - only the axes differ. `pearson_r_percent` is reported
alongside `pearson_r` because r is scale-dependent and each figure quotes the scale it draws.

### Reactivity changes

The state-vs-state family carries one more highlight, and it is the only place this notebook
reads another assay: `reactivity_change` flags every protein carrying a cysteine reactivity
change in `notebooks/03_reactivity`, so the panel asks whether a protein whose cysteines
changed reactivity also moved between the detergent fractions.

The set is the **union across D4A/D4C/D8A/D8C** (180 proteins, 136 of them detected here).
Unioned rather than state-matched: this run has no D4 states to match against, and a per-panel
highlight set would change the coloured points from panel to panel, which is not the comparison
being drawn. The join is on the gene symbol, not on `uniprot` - the reactivity table carries the
peptide's accession and this one the protein-level accession, and they disagree for isoform
entries often enough to lose proteins silently.

`reactivity_named` is what carries a text label - `REACTIVITY_NAMED_PROTEINS`: MAP2K3, MAP2K4,
LONP1, HSPA9, PSMC5, DNAJA3, NDUFA9, VDAC1, eight in all. It is not a subset of the coloured
category, since some of the eight carry a reactivity change and some do not.


In [ ]:
rev_reactivity = load_reactivity_changes()
rev_scatter_summary = run_scatters(
    rev_long_df, rev_scatter_dir, reactivity_proteins=rev_reactivity
)

# The two families are rotations of one difference; confirm it holds numerically, pairing
# rows by (condition, control) rather than float equality of median_delta.
def _state_pair(row):
    if row["kind"] == "state_vs_state":
        return (row["y_short"], row["x_short"])
    return tuple(part.split("-")[0] for part in row["x_label"].split("/"))


keyed = rev_scatter_summary.assign(
    state_pair=rev_scatter_summary.apply(_state_pair, axis=1)
).set_index(["state_pair", "kind"])

for pair in keyed.index.get_level_values("state_pair").unique():
    rotations = keyed.loc[pair]
    assert set(rotations.index) == {"state_vs_state", "soluble_vs_insoluble"}, pair
    a, b = rotations.loc["state_vs_state"], rotations.loc["soluble_vs_insoluble"]
    assert np.isclose(a["median_delta"], b["median_delta"]), pair
    for col in ["n_concordant", "n_higher_on_x", "n_higher_on_y"]:
        assert a[col] == b[col], (pair, col)

print(f"all {keyed.index.get_level_values('state_pair').nunique()} rotation pairs agree")

print(
    rev_scatter_summary[rev_scatter_summary["kind"] == "state_vs_state"][
        ["x_short", "y_short", "n_reactivity_change", "n_reactivity_source",
         "n_reactivity_named"]
    ].to_string(index=False)
)

rev_scatter_summary[
    ["kind", "x_short", "y_short", "n_joined", "pearson_r", "median_delta",
     "n_concordant", "n_higher_on_x", "n_higher_on_y"]
]

### An interactive companion

The static panel is three inches wide and names ten of its 4823 points, so 126 of the 136
reactivity-change proteins are anonymous dots on it. `run_reactivity_html` writes a plotly
version of the same three panels into the same folder, one `.html` beside each `.svg`/`.png`,
where hovering a point gives the protein, its accession, both percentages, the percentage-point
shift, its per-condition reactivity changes and its UniProt FUNCTION annotation. Clicking a
legend entry isolates a category, which is the quickest way to look at the 136 on their own.

These are exploration output, not manuscript figures - the SVG/PNG panels remain the figures of
record. Each file loads plotly.js from the CDN, so opening one needs a network connection.

The UniProt text comes from the `uniprot_functions.csv` cache `write_percent_insoluble_excel`
leaves behind, read directly rather than through `uniprot_functions()`: that helper re-queries the whole
set the moment one accession is missing, and one always is (the cache is built after
`drop_contaminants`, so it never holds `contaminant_INT-STD1`). 4590 of the 4823 carry a
function; the rest either have no FUNCTION comment in UniProt or are that contaminant.


In [ ]:
rev_reactivity_html = run_reactivity_html(rev_long_df, rev_scatter_dir)


## Fraction shares - the barplot input

Each protein's soluble and insoluble share of its state's total, one row per replicate
channel, so the two bars of a state sum to 100%. Soluble duplicate `i` is paired with
insoluble duplicate `i` of the same donor - the duplicate index carries no meaning of its own
across fractions, but it is the only pairing available, and it keeps every plotted point a
genuine share whose partner sums with it to 100.

Each bar therefore carries all four channels, two donors x two TMT duplicates, which the Rmd
draws as plain circles - donor is not shape-coded in this figure.

**These shares are relative.** Equal protein *mass* was labelled from each fraction and no
Final dilution factor exists for this run, so the channel ratio reports a protein's share
within its own fraction. A 90% soluble bar does not mean 90% of the protein is soluble.
Compare bars across states, not against 50.

Percent-of-control is a per-row rescaling of the channel ratio, so the constant divides out
of both numerator and denominator and the share is the same from either table.

Two CSVs, keyed by `BARPLOT_PROTEIN_SETS`: the manuscript's eight proteins
(`PROTEINS_OF_INTEREST` plus PSMC5, one of the 19S regulatory ATPase subunits), and
`POSITIVE_CONTROLS` - GZMB, HSP90B1, PRF1 - drawn as their own panel.

In [ ]:
rev_shares = fraction_shares(rev_filtered, cfg=rev_cfg, out_dir=rev_barplot_dir)

for name, panel in BARPLOT_PROTEIN_SETS.items():
    print(f"\n{name} - % insoluble")
    print(
        rev_shares[
            (rev_shares["protein"].isin(panel)) & (rev_shares["fraction"] == "Insoluble")
        ]
        .pivot_table(index="protein", columns="state", values="percent")
        .round(1)
        .to_string()
    )

In [ ]:
# A bar height is the mean of the per-channel shares; a scatter point is the share of the
# channel means - reconcile the two so the figures cannot quietly contradict each other.
rev_percent_agreement = check_percent_agreement(
    rev_shares, rev_long_df, out_dir=rev_barplot_dir
)
rev_percent_agreement["difference_pp"].abs().describe().round(3)

## GSEA

Same pre-ranked GO:BP analysis as HK3-124, on the three fraction contrasts. Positive NES =
enriched among proteins gaining intensity in the insoluble fraction. Takes about 10 minutes.

In [ ]:
rev_gsea_results = run_fraction_gsea(rev_long_df, rev_results_dir / "gsea")

for state, report in rev_gsea_results.items():
    fold_changes = (
        rev_long_df[
            (rev_long_df["condition"] == f"{state}-Insoluble")
            & (rev_long_df["control_condition"] == f"{state}-Soluble")
        ]
        .groupby("protein")["log2_FC"]
        .mean()
    )
    overall = fold_changes.median()
    # gseapy returns res2d columns as object dtype, so NES needs coercing before
    # nlargest/nsmallest.
    report = report.assign(NES=pd.to_numeric(report["NES"]))
    significant = report[pd.to_numeric(report["FDR q-val"]) < 0.01]

    def leading_edge_median(row):
        genes = [g for g in str(row["Lead_genes"]).split(";") if g in fold_changes.index]
        return fold_changes.loc[genes].median() if genes else np.nan

    top_up = significant.nlargest(5, "NES").apply(leading_edge_median, axis=1)
    top_down = significant.nsmallest(5, "NES").apply(leading_edge_median, axis=1)
    print(
        f"{state}: overall median log2(I/S) {overall:.2f}; "
        f"top +NES leading edge {top_up.median():.2f}, top -NES {top_down.median():.2f}"
    )
    assert top_up.median() > overall > top_down.median(), state

## Supplementary data table

The solubility data ships as a single sheet of `Data S2`, **`S2-6`**: percent insoluble per
replicate channel, plus the median per cell state, one row per protein. That is the quantity
the barplots and the percent-insoluble scatterplots draw, on the scale they draw it, so the
table and the figures cannot disagree.

Contaminant and keratin rows are dropped as in `low_input.write_percent_control_to_excel`,
leaving 4822 of the 4823 proteins, and every row carries the UniProt FUNCTION comment read
from the `uniprot_functions.csv` cache. A handful of channels are missing where converting a
TMT zero to `NaN` left no value; the medians are taken over whatever channels a state has.

The shares are relative - equal protein mass was labelled from each fraction and this run has
no Final dilution factor - so the sheet's description row says to compare across states rather
than against 50.

In [ ]:
rev_supp_table = write_percent_insoluble_excel(
    rev_shares,
    "../../supp_data/Data S2-6_solubility.xlsx",
    cfg=rev_cfg,
    cache_csv=rev_results_dir / "uniprot_functions.csv",
)

replicate_cols = [f"{state}_{rep}" for state in STATES for rep in rev_cfg.replicates]
annotated = (rev_supp_table["uniprot_function"].fillna("") != "").sum()
missing = int(rev_supp_table[replicate_cols].isna().sum().sum())

print(
    f"{S2_6_SHEET}: {rev_supp_table.shape[0]} proteins x {rev_supp_table.shape[1]} cols   "
    f"{annotated} annotated"
)
print(
    f"missing replicate channels: {missing} of "
    f"{len(rev_supp_table) * len(replicate_cols)} "
    f"({rev_supp_table[replicate_cols].isna().any(axis=1).sum()} proteins)"
)
rev_supp_table.head()